# T2SMark 官方参考真实复现入口

该 Notebook 用于 Colab GPU 冷启动: 挂载 Google Drive、拉取仓库、加载 SD3.5 Medium 所需依赖、运行 T2SMark 官方路径、生成统一 adapter observations、构造正式导入候选记录并打包保存到 Google Drive。

运行依赖: 不依赖主方法前序结果包, 但必须共享当前 `SLM_WM_PAPER_RUN_NAME` 派生的 prompt split、样本数和目标 FPR。该入口可与其他 official reference 入口并行运行, 最终由 `paper_result_closure_run.ipynb` 统一导入。

Notebook 通过 `paper_workflow/notebook_utils/notebook_entrypoint.py` 分派到 `paper_experiments/runners/t2smark_formal_reproduction.py`, 不承载正式实验逻辑。

## 运行前准备

1. 在 Colab 中选择 GPU runtime。
2. 确认 Hugging Face 账号已获得 `stabilityai/stable-diffusion-3.5-medium` 访问权限, 并配置 `HF_TOKEN`。
3. 默认使用 `probe_paper` 的全部70个 Prompt 和 FPR=0.1; 显式切换运行层级后, 使用该层级的完整 Prompt 与目标 FPR 复盘共同协议。
4. 本入口会写出正式导入候选记录, 但只有 fixed-FPR 与攻击矩阵边界同时满足时, validator 才会接受为当前运行层级的受治理论文证据。
5. 默认产物写入当前论文运行层级的 `external_baseline_official_reference` Drive 子目录。
6. Notebook 只准备 CPU `workflow_orchestrator`; repository workflow 会在运行时创建并核验 `t2smark_sd35_gpu` 隔离科学子解释器.

运行期间, repository 共享持久化会话会把稳定文件周期写入当前 workflow 的 Drive 目录, 并在恢复前复验 formal execution commit、科学 profile、配置身份与逐文件 SHA-256. checkpoint 仅用于续跑, 不属于正式论文证据。

In [ ]:
SLM_WM_PAPER_RUN_NAME = "probe_paper"

import os

from google.colab import drive

drive.mount('/content/drive')
# 修改为 "full_paper" 可切换完整论文运行层级。
os.environ["SLM_WM_PAPER_RUN_NAME"] = SLM_WM_PAPER_RUN_NAME


In [ ]:
import os
import re
import subprocess
import sys
from pathlib import Path

repository_url = os.environ.get("SLM_WM_REPOSITORY_URL", "https://github.com/RICHAAARC/SLM-WM.git")
repository_commit = os.environ.get("SLM_WM_REPOSITORY_COMMIT", "")
if re.fullmatch(r"[0-9a-f]{40}", repository_commit) is None:
    raise ValueError("正式运行前必须通过 SLM_WM_REPOSITORY_COMMIT 提供精确40位小写 Git SHA")
workspace_dir = Path(os.environ.get("SLM_WM_WORKSPACE_DIR", "/content/slm_wm_repository"))
workspace_dir.parent.mkdir(parents=True, exist_ok=True)
if workspace_dir.exists() and not (workspace_dir / ".git").is_dir():
    raise RuntimeError("SLM_WM_WORKSPACE_DIR 已存在但不是 Git 仓库")
if not (workspace_dir / ".git").is_dir():
    subprocess.run(["git", "clone", repository_url, str(workspace_dir)], check=True)

status_before_checkout = subprocess.run(
    ["git", "status", "--porcelain=v1", "--untracked-files=all"],
    cwd=workspace_dir,
    check=True,
    capture_output=True,
    text=True,
).stdout
if status_before_checkout:
    raise RuntimeError("复用 Colab 工作区前必须先清理 Git 工作树")
subprocess.run(["git", "fetch", "origin", "--tags", "--prune"], cwd=workspace_dir, check=True)
subprocess.run(["git", "rev-parse", "--verify", repository_commit + "^{commit}"], cwd=workspace_dir, check=True)
subprocess.run(["git", "checkout", "--detach", repository_commit], cwd=workspace_dir, check=True)

os.chdir(workspace_dir)
if str(workspace_dir) not in sys.path:
    sys.path.insert(0, str(workspace_dir))
from paper_workflow.colab_utils.formal_execution import verify_and_publish_formal_execution

formal_execution_lock = verify_and_publish_formal_execution(workspace_dir, repository_commit)
print({"workspace_dir": str(workspace_dir), "formal_execution_lock": formal_execution_lock})


In [ ]:
DEPENDENCY_PROFILE_ID = "workflow_orchestrator"
dependency_profile_result = subprocess.run(
    ["python", "scripts/prepare_dependency_profile.py", "--profile", DEPENDENCY_PROFILE_ID],
    check=True,
)
print({"dependency_profile_id": DEPENDENCY_PROFILE_ID, "return_code": dependency_profile_result.returncode})

from paper_workflow.colab_utils.dependency_check import build_notebook_dependency_report

dependency_report = build_notebook_dependency_report(DEPENDENCY_PROFILE_ID)
print(dependency_report)
assert dependency_report["dependency_decision"] == "pass", dependency_report


In [ ]:
from paper_workflow.colab_utils.paper_run_environment import configure_paper_run_environment
paper_run_environment = configure_paper_run_environment(
    workflow_name="official_reference_t2smark",
)
print(paper_run_environment)


In [ ]:
import os

try:
    from google.colab import userdata
    token_from_secret = userdata.get('HF_TOKEN')
except Exception:
    token_from_secret = None

if token_from_secret and not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = token_from_secret

print({'hf_token_available_for_scientific_child': bool(os.environ.get('HF_TOKEN'))})


In [ ]:
gpu_probe = subprocess.run(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
    check=False,
    capture_output=True,
    text=True,
)
print({'gpu_probe_return_code': gpu_probe.returncode, 'gpu_names': gpu_probe.stdout.strip()})
assert gpu_probe.returncode == 0 and gpu_probe.stdout.strip(), '需要 Colab GPU runtime'


In [ ]:
from paper_workflow.notebook_utils.notebook_entrypoint import run_workflow

t2smark_formal_summary = run_workflow(root='.', workflow_name="official_reference_t2smark")
t2smark_formal_summary


In [ ]:
import os
from pathlib import Path

summary_path = Path('outputs/t2smark_formal_reproduction') / os.environ['SLM_WM_PAPER_RUN_NAME'] / 't2smark_formal_reproduction_summary.json'
validation_path = Path('outputs/t2smark_formal_reproduction') / os.environ['SLM_WM_PAPER_RUN_NAME'] / 't2smark_formal_import_validation_report.json'
print(summary_path.read_text(encoding='utf-8') if summary_path.exists() else t2smark_formal_summary)
if validation_path.exists():
    print(validation_path.read_text(encoding='utf-8'))


In [ ]:
from paper_workflow.notebook_utils.notebook_entrypoint import package_workflow_outputs

archive_record = package_workflow_outputs(root='.', workflow_name="official_reference_t2smark")
archive_record.to_dict()


In [ ]:
import os
from pathlib import Path

output_dir = Path('outputs/t2smark_formal_reproduction') / os.environ['SLM_WM_PAPER_RUN_NAME']

for result_path in sorted(output_dir.glob('*.json')):
    if result_path.is_file():
        print(result_path)
        print(result_path.read_text(encoding='utf-8')[:4000])

assert t2smark_formal_summary['run_decision'] == 'pass', t2smark_formal_summary
print('drive_archive_path', archive_record.drive_archive_path)
